# NB-12 — SegFormer factorial, dropout controls, curriculum sensitivity (WP6, WP7, WP8)

**Question:** is the gain from the *curriculum* (the schedule) or just from *modality dropout* — and is consistency additive on the same backbone?
Everything here is **SegFormer MiT-B0 (3.72 M), Study-1 protocol** (Focal-Tversky, AdamW 3e-4, wd 1e-4, 45 epochs, batch 8, EMA 0.99, grad-clip 1, consistency λ = 0.5, selection on mean val IoU over the 3 regimes), seeds 42 / 1 / 2.

| Group | Runs (× 3 seeds) | Answers |
|---|---|---|
| **A_factorial** (9 runs) | `seg_full_ref` (v8 re-run of the paper's model), `seg_plain` (no dropout, no consistency), `seg_consonly` (no dropout, SAR-only consistency pass on every example) | the 2 × 2 table: the other two cells already exist (`segformer_noconsist`, `segformer_concat`) |
| **B_controls** (9) | `seg_uniform` (1/3, 1/3, 1/3), `seg_fixed_asym` (0.40, 0.45, 0.15), `seg_matched` (fixed at the curriculum's *cumulative* exposure, computed from the schedule) | is it the curriculum, or just dropout? |
| **C_sensitivity** (9) | `seg_term_sar60` (terminal 0.25/0.60/0.15), `seg_term_mild` (0.60/0.25/0.15), `seg_warm0` (no warm-up) | how much the 0.40/0.45/0.15 design choice matters |

All three groups use consistency λ = 0.5, so the only thing that differs from the full recipe is the **dropout schedule**.
Every run logs the **measured** regime shares per epoch (counted, not computed), per-chip confusion counts on val / test / **Bolivia**, full config, history and checkpoint — NB-16 never needs a retrain.

**Attach:** `lgmf_v8` (new, see below) · Sen1Floods11 · `lgmf-index` · optionally `lgmf-nb11-audit` (gives the two existing factorial cells and the old SegFormer numbers as references).
**Settings:** GPU T4 · Internet **On** (ImageNet encoder weights) · *Save & Run All (Commit)*.
**Budget:** ~25-30 min per run on a T4 → about 4-5 h per group. Run **one group per commit** (set `GROUPS_TO_RUN` below). Between commits, attach the previous commit's output as an input: the RESTORE cell copies finished runs forward, so every commit's output is cumulative and finished runs are never repeated. The loop also stops on its own before Kaggle's 12 h limit.

## 0. Bootstrap

In [ ]:
import sys, os, glob, json, time, shutil, warnings, re, importlib, dataclasses, copy
warnings.filterwarnings("ignore")
T_START = time.time()
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py",
            "losses.py", "ema.py", "cloud.py", "ramdata.py", "audit.py", "runner.py", "selftest_v8.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_cands = []
print("every lgmf/ folder found under /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)):
    d = os.path.dirname(p); miss = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not miss else 'missing ' + ','.join(miss)}  {d}")
    if not miss: _cands.append((_ver(p), d))
assert _cands, "attach the lgmf_v8 dataset (v7 and older do not have runner/ramdata/cloud/audit)"
_best = max(_cands); assert _best[0] >= (8,), f"best complete lgmf is v{_best[0]}; attach lgmf_v8"
PKG_DIR = _best[1]; sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
for mod, pipname in (("segmentation_models_pytorch", "segmentation-models-pytorch"),):
    try: __import__(mod)
    except ImportError: os.system(f"pip install -q {pipname}")
import numpy as np, pandas as pd, torch
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR, lgmf.__file__
from lgmf.config import LGMFConfig
from lgmf.curriculum import regime_probs
from lgmf.ramdata import RamDataset
from lgmf.runner import run_experiment, is_done, run_dir
from lgmf.audit import metrics_from_counts
print("lgmf", lgmf.__version__, "|", PKG_DIR)

ns = sorted(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)); assert ns, "attach lgmf-index"
idx = os.path.dirname(ns[0]); stats = json.load(open(ns[0]))
for s in ("train", "val", "test", "bolivia"): assert os.path.exists(f"{idx}/{s}_index.json"), s
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU: only for the smoke test)")

# ------------------------------ the knobs ------------------------------
EPOCHS = int(os.environ.get("NB12_EPOCHS", 45))        # never change mid-stream: results live in runs_e<EPOCHS>
SEEDS = [42, 1, 2]
GROUPS_TO_RUN = ["A_factorial"]                         # <- commit 1. Then ["B_controls"], then ["C_sensitivity"]
TIME_BUDGET_H = 10.5                                    # stop launching runs that would not finish inside this
OUT = "/kaggle/working"

## 1. Preflight (about 1 min): self-test of the package on synthetic data

In [ ]:
from lgmf import selftest_v8
selftest_v8.run(device=device)       # raises if anything in the v8 package is broken -- better now than after 2 hours

## 2. RESTORE — carry finished runs forward from a previous commit of this notebook

In [ ]:
def restore_previous():
    n = 0
    for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True):
        src = os.path.dirname(p); rel = src.split(f"runs_e{EPOCHS}" + os.sep, 1)[1]
        dst = os.path.join(OUT, f"runs_e{EPOCHS}", rel)
        if not os.path.exists(os.path.join(dst, "result.json")):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst); n += 1
    return n
print("restored", restore_previous(), "finished run(s) from attached inputs")
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")): print("  have", p.split(f"runs_e{EPOCHS}/")[1].rsplit("/result.json", 1)[0])

## 3. Data into RAM (once) and the run table

In [ ]:
base = LGMFConfig(); base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]; base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]
t0 = time.time()
TRAIN = RamDataset(base, f"{idx}/train_index.json", augment=True)
EVAL = {s: RamDataset(base, f"{idx}/{s}_index.json", augment=False) for s in ("val", "test", "bolivia")}
print({"train": len(TRAIN), **{k: len(v) for k, v in EVAL.items()}}, f"cached in {time.time()-t0:.0f}s")

def make_cfg(overrides, seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size, c.ema_decay = EPOCHS, 3e-4, 8, 0.99
    c.loss_type, c.consistency_weight, c.num_workers = "focal_tversky", 0.5, 0
    c.s1_mean, c.s1_std, c.s2_mean, c.s2_std = stats["s1_mean"], stats["s1_std"], stats["s2_mean"], stats["s2_std"]
    c.seed = seed
    for k, v in overrides.items(): setattr(c, k, v)
    return c

# matched exposure: the cumulative regime probabilities the curriculum actually applies over the run (read from the code's schedule)
_c = make_cfg({}, 0)
MATCHED = tuple(float(x) for x in np.mean([regime_probs(_c, e) for e in range(EPOCHS)], 0))
print("curriculum cumulative exposure (both, SAR-only, optical-only):", np.round(MATCHED, 4))

SEG = dict(kind="concat", arch="segformer", encoder="mit_b0")
NODROP = dict(use_curriculum=False, fixed_regime_probs=(1.0, 0.0, 0.0))
RUNS = [   # (run_id, group, config overrides)
    ("seg_full_ref",   "A_factorial",   {}),                                             # the paper's recipe, re-run with v8 logging
    ("seg_plain",      "A_factorial",   dict(NODROP, consistency_weight=0.0)),            # neither
    ("seg_consonly",   "A_factorial",   dict(NODROP)),                                    # consistency pass on every example, no dropout
    ("seg_uniform",    "B_controls",    dict(use_curriculum=False, fixed_regime_probs=(1/3, 1/3, 1/3))),
    ("seg_fixed_asym", "B_controls",    dict(use_curriculum=False, fixed_regime_probs=(0.40, 0.45, 0.15))),
    ("seg_matched",    "B_controls",    dict(use_curriculum=False, fixed_regime_probs=MATCHED)),
    ("seg_term_sar60", "C_sensitivity", dict(final_regime_probs=(0.25, 0.60, 0.15))),
    ("seg_term_mild",  "C_sensitivity", dict(final_regime_probs=(0.60, 0.25, 0.15))),
    ("seg_warm0",      "C_sensitivity", dict(warmup_epochs=0)),
]
display(pd.DataFrame([dict(run=r, group=g, curriculum=o.get("use_curriculum", True), fixed=o.get("fixed_regime_probs"),
                           terminal=o.get("final_regime_probs", "(0.40,0.45,0.15)" if o.get("use_curriculum", True) else "-"),
                           warmup=o.get("warmup_epochs", 5 if o.get("use_curriculum", True) else "-"),
                           consistency=o.get("consistency_weight", 0.5)) for r, g, o in RUNS]))

## 4. Train (seed-major, so every arm has seed 42 early; resumable; stops before the session limit)

In [ ]:
todo = [(rid, g, ov, sd) for sd in SEEDS for rid, g, ov in RUNS if g in GROUPS_TO_RUN and not is_done(OUT, EPOCHS, rid, sd)]
print(f"{len(todo)} run(s) to do in {GROUPS_TO_RUN}")
durations = []
for rid, g, ov, sd in todo:
    est = (np.mean(durations) if durations else 35.0) * 1.15
    left = TIME_BUDGET_H * 60 - (time.time() - T_START) / 60
    if est > left:
        print(f"STOP: next run needs ~{est:.0f} min, only {left:.0f} min of budget left. "
              f"Save this version, attach its output to a new commit, and run again (finished runs are skipped)."); break
    t1 = time.time()
    run_experiment(rid, make_cfg(ov, sd), SEG, TRAIN, EVAL["val"], EVAL, OUT, device, select="mean")
    durations.append((time.time() - t1) / 60)
print(f"done; {len(durations)} run(s) this session, {(time.time()-T_START)/3600:.1f} h elapsed")

## 5. Results — every run, test and Bolivia, mean ± std over seeds (ddof = 1; the paper's ddof = 0 understates spread at n = 3)
Reference rows `ref:*` come from the earlier runs (via `lgmf-nb11-audit`, if attached): `ref:segformer_noconsist` is the **curriculum-only** factorial cell, `ref:segformer_concat` the **full-recipe** cell.

In [ ]:
recs = []
for p in glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json"):
    r = json.load(open(p))
    for split in ("val", "test", "bolivia"):
        for reg, m in r[split].items():
            recs.append(dict(run=r["run_id"], seed=r["seed"], split=split, regime=reg, iou=m["iou"], flood_iou=m["flood_iou"]))
# references from NB-11's per-chip file, if attached
ref_files = glob.glob("/kaggle/input/**/per_chip_counts.csv.gz", recursive=True)
if ref_files:
    pc = pd.read_csv(ref_files[0]); pc = pc[(pc.variant == "zero") & pc.config.isin(["segformer_concat", "segformer_noconsist", "unet_drop", "manet_drop"])]
    g = pc.groupby(["config", "seed", "split", "regime"])[["tp", "fp", "fn", "tn", "ftp", "ffp", "ffn", "ftn"]].sum().reset_index()
    g["iou"] = metrics_from_counts(g.tp, g.fp, g.fn, g.tn)["iou"]; g["flood_iou"] = metrics_from_counts(g.ftp, g.ffp, g.ffn, g.ftn)["iou"]
    for r in g.itertuples(): recs.append(dict(run="ref:" + r.config, seed=int(r.seed), split=r.split, regime=r.regime, iou=r.iou, flood_iou=r.flood_iou))
    print("references loaded from", ref_files[0])
else:
    print("lgmf-nb11-audit not attached: no ref:* rows (the two existing factorial cells will be missing)")
R = pd.DataFrame(recs); R.to_csv(f"{OUT}/nb12_per_seed.csv", index=False)

def table(split):
    g = R[R.split == split].groupby(["run", "regime"]).iou.agg(["mean", "std", "count"]).reset_index()
    g["cell"] = g.apply(lambda r: f"{r['mean']:.3f}±{(r['std'] if r['std'] == r['std'] else 0):.3f}", axis=1)
    t = g.pivot(index="run", columns="regime", values="cell")[["both", "s1_only", "s2_only"]]
    t["n"] = g.groupby("run")["count"].max()
    return t
for sp in ("test", "bolivia"):
    print(f"\n=== {sp.upper()}  (IoU, mean ± std over seeds, ddof=1) ==="); display(table(sp))
table("test").to_csv(f"{OUT}/nb12_table_test.csv"); table("bolivia").to_csv(f"{OUT}/nb12_table_bolivia.csv")

## 6. Factorial effects and controls — paired by seed
Differences are computed seed by seed (same seed = same initialisation of the new layers and same data order stream), then summarised. With 3 seeds this is **descriptive** — read the per-seed column; a result that has the same sign in all 3 seeds and a mean larger than the seed spread is worth reporting, anything else is not a finding.

In [ ]:
def per_seed(run, split, regime, col="iou"):
    q = R[(R.run == run) & (R.split == split) & (R.regime == regime)].set_index("seed")[col]
    return q
CONTRASTS = [   # (label, A, B)  -> A minus B
    ("curriculum effect, no consistency",          "ref:segformer_noconsist", "seg_plain"),
    ("consistency effect, no curriculum",          "seg_consonly",            "seg_plain"),
    ("consistency effect, with curriculum",        "ref:segformer_concat",    "ref:segformer_noconsist"),
    ("curriculum effect, with consistency",        "ref:segformer_concat",    "seg_consonly"),
    ("full recipe vs uniform dropout",             "seg_full_ref",            "seg_uniform"),
    ("full recipe vs fixed asymmetric 0.40/0.45/0.15", "seg_full_ref",        "seg_fixed_asym"),
    ("full recipe vs fixed MATCHED exposure",      "seg_full_ref",            "seg_matched"),
    ("sensitivity: terminal SAR-heavy vs default", "seg_term_sar60",          "seg_full_ref"),
    ("sensitivity: terminal mild vs default",      "seg_term_mild",           "seg_full_ref"),
    ("sensitivity: no warm-up vs default",         "seg_warm0",               "seg_full_ref"),
    ("v8 re-run vs earlier run (reproducibility)", "seg_full_ref",            "ref:segformer_concat"),
]
rows = []
for lab, A, B in CONTRASTS:
    for sp in ("test", "bolivia"):
        for reg in ("both", "s1_only", "s2_only"):
            a, b = per_seed(A, sp, reg), per_seed(B, sp, reg); common = a.index.intersection(b.index)
            if len(common) == 0: continue
            d = (a[common] - b[common])
            rows.append(dict(contrast=lab, split=sp, regime=reg, n=len(common), mean_diff=d.mean(), seed_sd=d.std() if len(d) > 1 else np.nan,
                             per_seed=" ".join(f"{s}:{v:+.3f}" for s, v in d.items()), same_sign=bool((d > 0).all() or (d < 0).all())))
C = pd.DataFrame(rows)
if len(C):
    C.to_csv(f"{OUT}/nb12_contrasts.csv", index=False)
    pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 80)
    display(C.round(4))
# 2x2 interaction on SAR-only (needs all four cells)
cells = {"plain": "seg_plain", "cons": "seg_consonly", "curr": "ref:segformer_noconsist", "full": "ref:segformer_concat"}
for sp in ("test", "bolivia"):
    try:
        m = {k: per_seed(v, sp, "s1_only").mean() for k, v in cells.items()}
        if any(np.isnan(list(m.values()))): raise KeyError
        print(f"\n2x2 factorial, SAR-only IoU, {sp}: plain {m['plain']:.3f} | consistency-only {m['cons']:.3f} | curriculum-only {m['curr']:.3f} | both {m['full']:.3f}"
              f"   interaction = {(m['full']-m['curr'])-(m['cons']-m['plain']):+.3f}")
    except KeyError:
        print(f"2x2 ({sp}): not all four cells available yet")

## 7. Was the matched-exposure control actually matched? (measured, not computed)

In [ ]:
rows = []
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")):
    r = json.load(open(p))
    rows.append(dict(run=r["run_id"], seed=r["seed"], measured_both=r["measured_cumulative_shares"][0], measured_sar=r["measured_cumulative_shares"][1],
                     measured_opt=r["measured_cumulative_shares"][2], scheduled_both=r["scheduled_cumulative_probs"][0],
                     scheduled_sar=r["scheduled_cumulative_probs"][1], scheduled_opt=r["scheduled_cumulative_probs"][2]))
X = pd.DataFrame(rows)
if len(X):
    g = X.groupby("run")[[c for c in X.columns if c.startswith(("measured", "scheduled"))]].mean()
    display(g.round(4)); g.to_csv(f"{OUT}/nb12_measured_exposure.csv")
    if {"seg_full_ref", "seg_matched"} <= set(g.index):
        d = (g.loc["seg_matched", ["measured_both", "measured_sar", "measured_opt"]].values - g.loc["seg_full_ref", ["measured_both", "measured_sar", "measured_opt"]].values)
        print("measured cumulative exposure, matched control minus curriculum (both, SAR, optical):", np.round(d, 4),
              "-> matched" if np.abs(d).max() < 0.02 else "-> NOT matched, investigate")

## 8. What to send back
When a group has finished: **Save Version → Save & Run All**, then send me the printed tables of sections 5-7 (or the executed `.ipynb`).
Next commit: attach this notebook's previous output, change `GROUPS_TO_RUN` to the next group, and run again.
After group C: publish the output as the dataset `lgmf-nb12` (NB-16 needs the per-chip files in it).